# 05 データ可視化・ケモメトリクス診断

`04_preprocess_model_interval_ensemble_production.ipynb` のモデル構築に入る前に、データの構造を観察するための notebook です。

1. データ読み込み
2. データ可視化・ケモメトリクス診断
3. 目的変数分布、群差、スペクトル形状、前処理による変化、PCA、T2/Q residual、相関領域、外れ値候補の確認

モデル探索・波長区間選択・アンサンブルは 04 で実行します。05 は「モデルを回す前に、何が起きているデータなのかを見る」ための観察台として使います。


## 0. 設定と共通関数

In [ ]:
from __future__ import annotations

from datetime import datetime
from itertools import combinations
from pathlib import Path
import json
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
os.environ.setdefault('MPLCONFIGDIR', str(Path('/private/tmp') / 'matplotlib-cache'))
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_array, check_is_fitted

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')

for font_name in ['Hiragino Sans', 'AppleGothic', 'Apple SD Gothic Neo', 'Yu Gothic', 'Meiryo', 'Noto Sans CJK JP']:
    try:
        plt.rcParams['font.family'] = font_name
        break
    except Exception:
        pass
plt.rcParams['axes.unicode_minus'] = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path('/Users/ogawatomohiro/myproject')
DATA_DIR = PROJECT_ROOT / 'data'
BASE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'production'

DATA_PATH = DATA_DIR / 'production_dataset.csv'
SAMPLE_FALLBACK_PATH = DATA_DIR / 'train.csv'
ENCODINGS = ('cp932', 'utf-8-sig', 'utf-8', 'shift_jis')

TARGET_CANDIDATES = ['含水率', 'mc', 'moisture', 'moisture_content', 'water_content', 'target', 'y']
SAMPLE_CANDIDATES = ['サンプル名', 'サンプルID', 'sample name', 'sample_name', 'sample number', 'sample_id', 'id', 'ID']
GROUP_CANDIDATES = ['ロット', 'lot', 'batch', '樹種', 'species', 'species number', 'sample_name']

WAVELENGTH_RANGE_NM = (900, 1700)
VALID_SIZE = 0.20
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_SPLITS = 5
GROUP_SPLIT_COL = None

# 05は可視化・診断を主目的にします。モデル探索は04で実行します。
SPECTRAL_AXIS_UNIT = 'auto'  # auto, nm, cm-1
INVERT_WAVENUMBER_AXIS = True

PREFIX = '05_visual_chemometrics'
EXPERIMENT_NAME = None  # 例: 'wood_spectra_visual_check'。Noneなら初回実行時刻でRUN_IDを作ります。
if EXPERIMENT_NAME:
    RUN_ID = EXPERIMENT_NAME
elif 'RUN_ID' not in globals():
    RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

OUTPUT_DIR = BASE_OUTPUT_DIR / PREFIX / RUN_ID
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print('RUN_ID:', RUN_ID)
print('OUTPUT_DIR:', OUTPUT_DIR)


def json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    try:
        json.dumps(value)
        return value
    except TypeError:
        return repr(value)


def write_run_config(extra=None):
    payload = {
        'run_id': RUN_ID,
        'experiment_name': EXPERIMENT_NAME,
        'output_dir': OUTPUT_DIR,
        'data_path': DATA_PATH if DATA_PATH.exists() else SAMPLE_FALLBACK_PATH,
        'random_state': RANDOM_STATE,
        'n_splits': N_SPLITS,
        'valid_size': VALID_SIZE,
        'test_size': TEST_SIZE,
        'wavelength_range_nm': WAVELENGTH_RANGE_NM,
        'spectral_axis_unit': SPECTRAL_AXIS_UNIT,
        'invert_wavenumber_axis': INVERT_WAVENUMBER_AXIS,
        'group_split_col': GROUP_SPLIT_COL,
        'notebook_role': 'visualization_and_chemometrics_diagnostics_only',
        'modeling_notebook': '04_preprocess_model_interval_ensemble_production.ipynb',
    }
    if extra:
        payload.update(extra)
    path = OUTPUT_DIR / f'{PREFIX}_run_config.json'
    path.write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2), encoding='utf-8')
    return path

write_run_config()


def read_csv_with_fallback(path: Path, encodings=ENCODINGS) -> pd.DataFrame:
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def pick_first_existing(columns, candidates, required=False, label='column'):
    normalized = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in normalized:
            return normalized[key]
    if required:
        raise ValueError(f'{label} not found. candidates={candidates}')
    return None


def parse_wavelength_from_col(col):
    text = str(col).strip()
    if text in {'', 'nan'}:
        return None
    nums = re.findall(r'(?<!\d)(\d{3,5}(?:\.\d+)?)(?!\d)', text)
    if not nums:
        return None
    values = [float(x) for x in nums]
    lo, hi = WAVELENGTH_RANGE_NM if WAVELENGTH_RANGE_NM is not None else (0, float('inf'))
    in_range = [v for v in values if lo <= v <= hi]
    return in_range[-1] if in_range else values[-1]


def detect_spectral_columns(df: pd.DataFrame, exclude_cols=None, wavelength_range_nm=WAVELENGTH_RANGE_NM):
    exclude = set(exclude_cols or [])
    records = []
    for col in df.columns:
        if col in exclude:
            continue
        wl = parse_wavelength_from_col(col)
        if wl is None:
            continue
        numeric = pd.to_numeric(df[col], errors='coerce')
        if numeric.notna().mean() < 0.90:
            continue
        records.append((col, wl))
    if not records:
        raise ValueError('波長列を検出できませんでした。列名に 900, 901nm, Abs_900 などの波長値が含まれているか確認してください。')
    spec_info = pd.DataFrame(records, columns=['column', 'wavelength_nm']).drop_duplicates('column')
    if wavelength_range_nm is not None:
        lo, hi = wavelength_range_nm
        ranged = spec_info.query('@lo <= wavelength_nm <= @hi').copy()
        if len(ranged) >= 10:
            spec_info = ranged
    spec_info = spec_info.sort_values('wavelength_nm').reset_index(drop=True)
    return spec_info['column'].tolist(), spec_info


def make_y_bins(y, max_bins=8):
    y = pd.Series(y).astype(float)
    n_bins = min(max_bins, max(2, len(y) // 10))
    try:
        bins = pd.qcut(y, q=n_bins, duplicates='drop')
        counts = bins.value_counts()
        if len(counts) >= 2 and counts.min() >= 2:
            return bins.astype(str)
    except Exception:
        pass
    return None


def split_single_dataset(df, target_col, group_col=None, valid_size=VALID_SIZE, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    df = df.dropna(subset=[target_col]).reset_index(drop=True).copy()
    if group_col is not None and group_col in df.columns and df[group_col].nunique() >= 3:
        splitter1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_valid_idx, test_idx = next(splitter1.split(df, groups=df[group_col]))
        train_valid = df.iloc[train_valid_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)
        rel_valid = valid_size / (1 - test_size)
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=rel_valid, random_state=random_state)
        train_idx, valid_idx = next(splitter2.split(train_valid, groups=train_valid[group_col]))
        return train_valid.iloc[train_idx].reset_index(drop=True), train_valid.iloc[valid_idx].reset_index(drop=True), test_df

    stratify = make_y_bins(df[target_col])
    train_valid, test_df = train_test_split(df, test_size=test_size, random_state=random_state, shuffle=True, stratify=stratify)
    rel_valid = valid_size / (1 - test_size)
    stratify_tv = make_y_bins(train_valid[target_col])
    train_df, valid_df = train_test_split(train_valid, test_size=rel_valid, random_state=random_state, shuffle=True, stratify=stratify_tv)
    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred):
    return {
        'r2': float(r2_score(y_true, y_pred)),
        'rmse': rmse(y_true, y_pred),
        'mae': float(mean_absolute_error(y_true, y_pred)),
    }


def odd_window(window_length, n_points):
    w = int(window_length)
    if w % 2 == 0:
        w += 1
    max_w = n_points if n_points % 2 == 1 else n_points - 1
    return max(3, min(w, max_w))


def load_production_dataset():
    path = DATA_PATH if DATA_PATH.exists() else SAMPLE_FALLBACK_PATH
    if path == SAMPLE_FALLBACK_PATH:
        print(f'NOTE: {DATA_PATH} がないため、動作確認用に {SAMPLE_FALLBACK_PATH} を読み込みます。')
    df = read_csv_with_fallback(path)
    target_col = pick_first_existing(df.columns, TARGET_CANDIDATES, required=True, label='target column')
    sample_col = pick_first_existing(df.columns, SAMPLE_CANDIDATES, required=False, label='sample id column')
    group_col = GROUP_SPLIT_COL or pick_first_existing(df.columns, GROUP_CANDIDATES, required=False, label='diagnostic group column')
    exclude_cols = [c for c in [target_col, sample_col, group_col] if c is not None]
    spectral_cols, spec_info = detect_spectral_columns(df, exclude_cols=exclude_cols)
    df[spectral_cols] = df[spectral_cols].apply(pd.to_numeric, errors='coerce')
    train_df, valid_df, test_df = split_single_dataset(df, target_col=target_col, group_col=GROUP_SPLIT_COL)
    config = {
        'data_path': str(path),
        'target_col': target_col,
        'sample_col': sample_col,
        'diagnostic_group_col': group_col,
        'group_split_col': GROUP_SPLIT_COL,
        'n_spectral_cols': len(spectral_cols),
        'spectral_axis_min': float(spec_info['wavelength_nm'].min()),
        'spectral_axis_max': float(spec_info['wavelength_nm'].max()),
        'spectral_axis_unit': SPECTRAL_AXIS_UNIT,
        'train_shape': train_df.shape,
        'valid_shape': valid_df.shape,
        'test_shape': test_df.shape,
    }
    return df, train_df, valid_df, test_df, spectral_cols, spec_info, config


def get_xy(part_df, spectral_cols, target_col):
    return part_df[spectral_cols].astype(float), part_df[target_col].astype(float)


def sample_ids(df, sample_col):
    if sample_col is not None and sample_col in df.columns:
        return df[sample_col].values
    return np.arange(len(df))


class SavitzkyGolayTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, window_length=21, polyorder=2, deriv=0, mode='interp'):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv
        self.mode = mode

    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.window_length_ = odd_window(self.window_length, self.n_features_in_)
        self.polyorder_ = min(int(self.polyorder), self.window_length_ - 1)
        return self

    def transform(self, X):
        check_is_fitted(self, ['n_features_in_', 'window_length_', 'polyorder_'])
        X = check_array(X, dtype=float)
        return savgol_filter(X, window_length=self.window_length_, polyorder=self.polyorder_, deriv=self.deriv, axis=1, mode=self.mode)


class SNVTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, ['n_features_in_'])
        X = check_array(X, dtype=float)
        mean = X.mean(axis=1, keepdims=True)
        std = X.std(axis=1, keepdims=True)
        std[std == 0] = 1.0
        return (X - mean) / std


class MSCTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.reference_ = X.mean(axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self, ['reference_'])
        X = check_array(X, dtype=float)
        corrected = np.empty_like(X, dtype=float)
        ref = self.reference_
        for i, row in enumerate(X):
            slope, intercept = np.polyfit(ref, row, deg=1)
            corrected[i] = row - intercept if abs(slope) < 1e-12 else (row - intercept) / slope
        return corrected


class AdaptivePLSRegression(BaseEstimator, RegressorMixin):
    def __init__(self, n_components=10, scale=True, max_iter=500, tol=1e-06):
        self.n_components = n_components
        self.scale = scale
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, X, y):
        X = check_array(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_components = max(1, min(int(self.n_components), X.shape[0] - 1, X.shape[1]))
        self.n_components_ = n_components
        self.model_ = PLSRegression(n_components=n_components, scale=self.scale, max_iter=self.max_iter, tol=self.tol)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        check_is_fitted(self, ['model_'])
        return self.model_.predict(X).ravel()


class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        self.columns_ = np.asarray(self.columns, dtype=int)
        return self

    def transform(self, X):
        X = check_array(X, dtype=float)
        return X[:, self.columns_]


## 1. データ読み込み

In [ ]:
df, train_df, valid_df, test_df, spectral_cols, spec_info, config = load_production_dataset()
target_col = config['target_col']
sample_col = config['sample_col']
group_col = config['diagnostic_group_col']
wavelengths = spec_info['wavelength_nm'].to_numpy(dtype=float)

X_train, y_train = get_xy(train_df, spectral_cols, target_col)
X_valid, y_valid = get_xy(valid_df, spectral_cols, target_col)
X_test, y_test = get_xy(test_df, spectral_cols, target_col)

print(json.dumps(config, ensure_ascii=False, indent=2, default=str))
display(spec_info.head())
display(pd.DataFrame({'split': ['train', 'valid', 'test'], 'n': [len(train_df), len(valid_df), len(test_df)], 'target_mean': [y_train.mean(), y_valid.mean(), y_test.mean()], 'target_std': [y_train.std(), y_valid.std(), y_test.std()]}))


## 2. データ可視化・ケモメトリクス診断

モデル探索に入る前に、目的変数分布、群差、スペクトル形状、前処理の効き方、PCAスコア、ローディング、波長ごとの相関を確認します。ここで見える外れ値・群差・散乱・ベースラインの傾向は、前処理やモデル選択の根拠になります。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

wavelengths = spec_info['wavelength_nm'].to_numpy(dtype=float)
X_all = df[spectral_cols].astype(float)
y_all = df[target_col].astype(float)


def infer_spectral_axis_unit(values, configured='auto'):
    if configured != 'auto':
        return configured
    values = np.asarray(values, dtype=float)
    return 'cm-1' if np.nanmedian(values) > 2500 else 'nm'


spectral_axis_unit = infer_spectral_axis_unit(wavelengths, SPECTRAL_AXIS_UNIT)
spectral_axis_label = 'Wavelength (nm)' if spectral_axis_unit == 'nm' else 'Wavenumber (cm$^{-1}$)'


def format_spectral_axis(ax):
    ax.set_xlabel(spectral_axis_label)
    if spectral_axis_unit == 'cm-1' and INVERT_WAVENUMBER_AXIS:
        ax.invert_xaxis()
    return ax


# 目的変数と群差
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(y_all, kde=True, ax=axes[0])
axes[0].set_title('Target distribution')
axes[0].set_xlabel(target_col)
if group_col is not None and group_col in df.columns:
    group_counts = df[group_col].astype(str).value_counts()
    top_groups = group_counts.head(12).index
    plot_df = df[df[group_col].astype(str).isin(top_groups)].copy()
    sns.boxplot(data=plot_df, x=group_col, y=target_col, ax=axes[1])
    axes[1].tick_params(axis='x', rotation=35)
    axes[1].set_title(f'{target_col} by {group_col}')
else:
    axes[1].axis('off')
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_target_distribution_and_group.png', dpi=160)

# 生スペクトルの重ね描きと平均±標準偏差
rng = np.random.default_rng(RANDOM_STATE)
sample_n = min(180, len(X_all))
sample_idx = rng.choice(len(X_all), size=sample_n, replace=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(wavelengths, X_all.iloc[sample_idx].T, color='#4c78a8', alpha=0.16, linewidth=0.75)
axes[0].set_title(f'Raw spectra overlay (n={sample_n})')
axes[0].set_ylabel('Signal')
format_spectral_axis(axes[0])
mean_spectrum = X_all.mean(axis=0).to_numpy(float)
std_spectrum = X_all.std(axis=0).to_numpy(float)
axes[1].plot(wavelengths, mean_spectrum, color='black', label='mean')
axes[1].fill_between(wavelengths, mean_spectrum - std_spectrum, mean_spectrum + std_spectrum, alpha=0.25, label='±1 std')
axes[1].set_title('Mean spectrum and variation')
axes[1].set_ylabel('Signal')
axes[1].legend()
format_spectral_axis(axes[1])
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_raw_spectra_overview.png', dpi=160)

# 前処理がスペクトル形状へ与える影響
preprocessing_preview = {
    'raw': [],
    'savgol_smooth_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=0))],
    'savgol_1st_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=1))],
    'savgol_2nd_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
    'snv': [('snv', SNVTransformer())],
    'snv_savgol_2nd_21': [('snv', SNVTransformer()), ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
    'msc_savgol_2nd_21': [('msc', MSCTransformer()), ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
}
preview_rows = []
fig, axes = plt.subplots(len(preprocessing_preview), 1, figsize=(12, 2.1 * len(preprocessing_preview)), sharex=True)
if len(preprocessing_preview) == 1:
    axes = [axes]
for ax, (prep_name, steps) in zip(axes, preprocessing_preview.items()):
    pipe = Pipeline(steps) if steps else None
    X_trans = X_all.to_numpy(float) if pipe is None else pipe.fit_transform(X_all)
    mean_trans = np.nanmean(X_trans, axis=0)
    ax.plot(wavelengths, mean_trans, color='#222222', linewidth=1.1)
    ax.set_title(prep_name, loc='left')
    ax.set_ylabel('Mean signal')
    format_spectral_axis(ax)
    preview_rows.append(pd.DataFrame({'preprocessing': prep_name, 'spectral_axis': wavelengths, 'mean_signal': mean_trans}))
preprocessing_preview_df = pd.concat(preview_rows, ignore_index=True)
preprocessing_preview_df.to_csv(OUTPUT_DIR / f'{PREFIX}_preprocessing_preview_mean_spectra.csv', index=False, encoding='utf-8-sig')
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_preprocessing_preview_mean_spectra.png', dpi=160)

# PCAスコア・ローディング・外れ値候補
X_scaled = StandardScaler().fit_transform(X_all)
pca = PCA(n_components=min(5, X_scaled.shape[0], X_scaled.shape[1]), random_state=RANDOM_STATE)
scores = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame(scores[:, :2], columns=['PC1', 'PC2'])
pca_df[target_col] = y_all.values
if group_col is not None and group_col in df.columns:
    pca_df[group_col] = df[group_col].astype(str).values
pca_df['sample_id'] = sample_ids(df, sample_col)
pca_df['score_distance'] = np.sqrt((pca_df['PC1'] / np.std(pca_df['PC1'])) ** 2 + (pca_df['PC2'] / np.std(pca_df['PC2'])) ** 2)
pca_df.sort_values('score_distance', ascending=False).head(30).to_csv(OUTPUT_DIR / f'{PREFIX}_pca_outlier_candidates.csv', index=False, encoding='utf-8-sig')
pca_df.to_csv(OUTPUT_DIR / f'{PREFIX}_pca_scores.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue=target_col, palette='viridis', s=32, alpha=0.85, ax=axes[0])
axes[0].set_title(f'PCA score plot by {target_col}')
if group_col is not None and group_col in pca_df.columns:
    sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue=group_col, s=32, alpha=0.75, ax=axes[1], legend=False)
    axes[1].set_title(f'PCA score plot by {group_col}')
else:
    axes[1].axis('off')
for ax in axes:
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_pca_scores.png', dpi=160)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(wavelengths, pca.components_[0], label='PC1 loading')
if pca.n_components_ >= 2:
    ax.plot(wavelengths, pca.components_[1], label='PC2 loading')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('PCA loadings')
ax.set_ylabel('Loading')
ax.legend()
format_spectral_axis(ax)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_pca_loadings.png', dpi=160)

# 波長ごとの目的変数相関
corr_raw = X_all.apply(lambda s: s.corr(y_all), axis=0).to_numpy(float)
X_deriv2 = SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2).fit_transform(X_all)
corr_deriv2 = pd.DataFrame(X_deriv2).apply(lambda s: s.corr(y_all), axis=0).to_numpy(float)
corr_df = pd.DataFrame({'spectral_axis': wavelengths, 'corr_raw': corr_raw, 'corr_2nd_derivative': corr_deriv2})
corr_df['abs_corr_raw'] = corr_df['corr_raw'].abs()
corr_df['abs_corr_2nd_derivative'] = corr_df['corr_2nd_derivative'].abs()
corr_df.to_csv(OUTPUT_DIR / f'{PREFIX}_spectral_axis_target_correlation.csv', index=False, encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(wavelengths, corr_raw, label='raw')
ax.plot(wavelengths, corr_deriv2, label='2nd derivative')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'Correlation with {target_col}')
ax.set_ylabel('Pearson correlation')
ax.legend()
format_spectral_axis(ax)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_target_correlation_by_spectral_axis.png', dpi=160)

# 目的変数順ヒートマップ
order = y_all.sort_values().index
heat = X_all.loc[order].to_numpy(float)
fig, ax = plt.subplots(figsize=(11, 5.5))
im = ax.imshow(heat, aspect='auto', extent=[wavelengths.min(), wavelengths.max(), len(order), 0], cmap='viridis')
ax.set_title('Spectra heatmap sorted by target')
ax.set_ylabel('Samples sorted by target')
format_spectral_axis(ax)
fig.colorbar(im, ax=ax, label='Signal')
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_spectra_heatmap_sorted_by_target.png', dpi=160)

### 2.1 追加のケモメトリクス診断

モデル構築前の意思決定に使いやすいよう、split後の目的変数分布、目的変数ビン別平均スペクトル、PCAの寄与率・T2/Q residual、群平均との差、相関が強いスペクトル領域を追加で確認します。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# splitごとの目的変数分布。splitが偏るとvalid/test評価の読み方が変わります。
split_target_df = pd.concat([
    pd.DataFrame({'split': 'train', target_col: train_df[target_col].astype(float).values}),
    pd.DataFrame({'split': 'valid', target_col: valid_df[target_col].astype(float).values}),
    pd.DataFrame({'split': 'test', target_col: test_df[target_col].astype(float).values}),
], ignore_index=True)
split_target_summary = split_target_df.groupby('split')[target_col].agg(['count', 'mean', 'std', 'min', 'median', 'max']).reset_index()
split_target_summary.to_csv(OUTPUT_DIR / f'{PREFIX}_split_target_summary.csv', index=False, encoding='utf-8-sig')
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.kdeplot(data=split_target_df, x=target_col, hue='split', common_norm=False, fill=False, ax=axes[0])
axes[0].set_title('Target distribution by split')
sns.boxplot(data=split_target_df, x='split', y=target_col, ax=axes[1])
axes[1].set_title('Target range by split')
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_split_target_distribution.png', dpi=160)
display(split_target_summary)

# 目的変数ビン別の平均スペクトル。濃度/含水率に沿って系統的に変わる波長領域を見ます。
target_bins = pd.qcut(y_all, q=min(5, max(2, y_all.nunique())), duplicates='drop')
binned_spectra_rows = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, idx in pd.Series(target_bins, index=X_all.index).groupby(target_bins, observed=True).groups.items():
    mean_raw = X_all.loc[list(idx)].mean(axis=0).to_numpy(float)
    mean_deriv2 = X_deriv2[list(idx)].mean(axis=0)
    axes[0].plot(wavelengths, mean_raw, label=str(label), linewidth=1.2)
    axes[1].plot(wavelengths, mean_deriv2, label=str(label), linewidth=1.2)
    binned_spectra_rows.append(pd.DataFrame({'target_bin': str(label), 'spectral_axis': wavelengths, 'mean_raw': mean_raw, 'mean_2nd_derivative': mean_deriv2}))
binned_spectra_df = pd.concat(binned_spectra_rows, ignore_index=True)
binned_spectra_df.to_csv(OUTPUT_DIR / f'{PREFIX}_target_binned_mean_spectra.csv', index=False, encoding='utf-8-sig')
axes[0].set_title('Mean raw spectra by target bin')
axes[0].set_ylabel('Signal')
axes[1].set_title('Mean 2nd-derivative spectra by target bin')
axes[1].set_ylabel('2nd derivative')
for ax in axes:
    ax.legend(title='target bin', fontsize=8)
    format_spectral_axis(ax)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_target_binned_mean_spectra.png', dpi=160)

# PCA scree、Hotelling T2、Q residual。スペクトル空間の外れ値候補を別角度から見ます。
n_pca_diag = min(10, X_scaled.shape[0], X_scaled.shape[1])
pca_diag = PCA(n_components=n_pca_diag, random_state=RANDOM_STATE)
scores_diag = pca_diag.fit_transform(X_scaled)
X_recon = pca_diag.inverse_transform(scores_diag)
q_residual = np.sum((X_scaled - X_recon) ** 2, axis=1)
score_var = np.var(scores_diag, axis=0, ddof=1)
score_var[score_var == 0] = 1.0
hotelling_t2 = np.sum((scores_diag ** 2) / score_var, axis=1)
pca_diag_df = pd.DataFrame({
    'sample_id': sample_ids(df, sample_col),
    target_col: y_all.values,
    'hotelling_t2': hotelling_t2,
    'q_residual': q_residual,
    'score_distance_pc1_pc2': pca_df['score_distance'].values,
})
if group_col is not None and group_col in df.columns:
    pca_diag_df[group_col] = df[group_col].astype(str).values
pca_diag_df['hotelling_t2_rank'] = pca_diag_df['hotelling_t2'].rank(ascending=False, method='first')
pca_diag_df['q_residual_rank'] = pca_diag_df['q_residual'].rank(ascending=False, method='first')
pca_diag_df['combined_outlier_rank'] = pca_diag_df[['hotelling_t2_rank', 'q_residual_rank']].mean(axis=1)
pca_diag_df.sort_values('combined_outlier_rank').head(40).to_csv(OUTPUT_DIR / f'{PREFIX}_pca_t2_q_outlier_candidates.csv', index=False, encoding='utf-8-sig')
pca_diag_df.to_csv(OUTPUT_DIR / f'{PREFIX}_pca_t2_q_residuals.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
components = np.arange(1, n_pca_diag + 1)
axes[0].bar(components, pca_diag.explained_variance_ratio_ * 100, color='#4c78a8')
axes[0].plot(components, np.cumsum(pca_diag.explained_variance_ratio_) * 100, color='black', marker='o')
axes[0].set_xticks(components)
axes[0].set_xlabel('PC')
axes[0].set_ylabel('Explained variance (%)')
axes[0].set_title('PCA scree and cumulative variance')
sns.scatterplot(data=pca_diag_df, x='hotelling_t2', y='q_residual', hue=target_col, palette='viridis', s=30, alpha=0.85, ax=axes[1])
axes[1].set_title('Hotelling T2 vs Q residual')
axes[1].set_xlabel('Hotelling T2')
axes[1].set_ylabel('Q residual')
if group_col is not None and group_col in pca_diag_df.columns:
    sns.scatterplot(data=pca_diag_df, x='hotelling_t2', y='q_residual', hue=group_col, s=30, alpha=0.75, legend=False, ax=axes[2])
    axes[2].set_title(f'T2 vs Q residual by {group_col}')
else:
    axes[2].axis('off')
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_pca_scree_t2_q_residual.png', dpi=160)

# 群平均との差スペクトル。樹種/ロットなどの系統差がどの領域に出るか確認します。
if group_col is not None and group_col in df.columns:
    top_groups = df[group_col].astype(str).value_counts().head(8).index
    global_mean = X_all.mean(axis=0).to_numpy(float)
    group_diff_rows = []
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for g in top_groups:
        idx = df[group_col].astype(str) == g
        group_mean = X_all.loc[idx].mean(axis=0).to_numpy(float)
        diff = group_mean - global_mean
        ax.plot(wavelengths, diff, label=str(g), linewidth=1.1)
        group_diff_rows.append(pd.DataFrame({'group': str(g), 'spectral_axis': wavelengths, 'mean_minus_global_mean': diff, 'n': int(idx.sum())}))
    group_diff_df = pd.concat(group_diff_rows, ignore_index=True)
    group_diff_df.to_csv(OUTPUT_DIR / f'{PREFIX}_group_mean_difference_spectra.csv', index=False, encoding='utf-8-sig')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Group mean spectra minus global mean: {group_col}')
    ax.set_ylabel('Mean difference')
    ax.legend(fontsize=8, ncol=2)
    format_spectral_axis(ax)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f'{PREFIX}_group_mean_difference_spectra.png', dpi=160)

# 相関の強い点と、近傍平均で強い領域。単一点ではなく領域として見るための補助です。
corr_region_df = corr_df.copy()
for col in ['abs_corr_raw', 'abs_corr_2nd_derivative']:
    corr_region_df[f'{col}_rolling_mean_21'] = corr_region_df[col].rolling(window=21, center=True, min_periods=5).mean()
top_corr_points = pd.concat([
    corr_region_df.nlargest(30, 'abs_corr_raw').assign(source='raw_point'),
    corr_region_df.nlargest(30, 'abs_corr_2nd_derivative').assign(source='derivative_point'),
], ignore_index=True)
top_corr_regions = pd.concat([
    corr_region_df.nlargest(30, 'abs_corr_raw_rolling_mean_21').assign(source='raw_region'),
    corr_region_df.nlargest(30, 'abs_corr_2nd_derivative_rolling_mean_21').assign(source='derivative_region'),
], ignore_index=True)
top_corr_points.to_csv(OUTPUT_DIR / f'{PREFIX}_top_correlation_points.csv', index=False, encoding='utf-8-sig')
top_corr_regions.to_csv(OUTPUT_DIR / f'{PREFIX}_top_correlation_regions.csv', index=False, encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(wavelengths, corr_region_df['abs_corr_raw_rolling_mean_21'], label='raw |corr| rolling mean')
ax.plot(wavelengths, corr_region_df['abs_corr_2nd_derivative_rolling_mean_21'], label='2nd derivative |corr| rolling mean')
ax.set_title(f'Local correlation strength with {target_col}')
ax.set_ylabel('Rolling mean absolute correlation')
ax.legend()
format_spectral_axis(ax)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_local_correlation_strength.png', dpi=160)
display(top_corr_regions.head(20))

# 外れ値候補のスペクトルを重ねて見る。PCA上で遠いサンプルが測定異常か、自然な群差かを確認します。
outlier_ids = pca_diag_df.sort_values('combined_outlier_rank').head(12)['sample_id'].tolist()
sample_id_values = pd.Series(sample_ids(df, sample_col), index=df.index)
outlier_mask = sample_id_values.isin(outlier_ids)
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(wavelengths, X_all.T, color='#cccccc', alpha=0.05, linewidth=0.5)
for idx in X_all.index[outlier_mask]:
    ax.plot(wavelengths, X_all.loc[idx], linewidth=1.3, label=str(sample_id_values.loc[idx]))
ax.set_title('PCA T2/Q outlier candidate spectra')
ax.set_ylabel('Signal')
ax.legend(fontsize=7, ncol=2)
format_spectral_axis(ax)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f'{PREFIX}_outlier_candidate_spectra.png', dpi=160)

## 3. モデル構築への引き継ぎ

この notebook で確認した外れ値候補、群差、相関が強い領域、前処理による形状変化をもとに、モデル構築は `04_preprocess_model_interval_ensemble_production.ipynb` で実行します。05 の出力CSVと図は `outputs/production/05_visual_chemometrics/{RUN_ID}/` にまとまります。
